<a href="https://colab.research.google.com/github/bangaru01/C_programing/blob/main/TS_ring_pucker_CREST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TS Ring-Pucker Conformer Search (CREST + Cremer–Pople)

Workflow:
1. Install `xtb` + `CREST`
2. Mount Google Drive (so you can read your 34 XYZ files and save results back)
3. Set which atoms define the ring and the forming bond
4. Run a **constrained** CREST conformer search (forming bond held fixed, everything else sampled freely)
5. Classify every conformer CREST finds (Chair / Boat / Twist-boat / Envelope / Half-chair) via formal Cremer–Pople analysis

**Important:** CREST here returns *constrained local minima*, not genuine transition states (no imaginary-frequency check).
Use these as starting geometries for real TS optimizations afterward — this step is for finding
which ring-pucker families are even worth optimizing.


## 1. Install xtb + CREST

In [ ]:
!apt-get -qq update
!apt-get -qq install -y xtb
!xtb --version

!wget -q -O crest.tar.xz "https://github.com/crest-lab/crest/releases/download/latest/crest-gnu-12-ubuntu-latest.tar.xz"
!tar xf crest.tar.xz
!chmod +x crest/crest
!./crest/crest --version


## 2. Mount Google Drive

This will prompt you to authorize access, then your entire Drive is available at `/content/drive/MyDrive/...`

Put your 34 XYZ files in a folder in your Drive first (e.g. `MyDrive/TS_structures/`), then point
`DRIVE_FOLDER` below at it. Results (CSVs) will also be written back into that same folder so they
persist after the Colab runtime disconnects — anything saved only to `/content/...` is lost when the
session ends, so always write final outputs under `/content/drive/MyDrive/...`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_FOLDER = "/content/drive/MyDrive/TS_structures"   # <-- EDIT to your actual folder
RESULTS_FOLDER = "/content/drive/MyDrive/TS_structures/results"
os.makedirs(RESULTS_FOLDER, exist_ok=True)

print("Files found in DRIVE_FOLDER:")
for f in sorted(os.listdir(DRIVE_FOLDER)):
    print(" ", f)


## 3. Pick a single structure to start with

(Either from Drive, or upload one directly if you just want to test the pipeline first.)

In [ ]:
# Option A: use a file already in your mounted Drive folder
xyz_path = os.path.join(DRIVE_FOLDER, "RR_tBu.xyz")   # <-- EDIT filename

# Option B: upload one manually instead (comment out Option A, uncomment this)
# from google.colab import files
# uploaded = files.upload()
# xyz_path = list(uploaded.keys())[0]

print("Using structure:", xyz_path)


## 4. Configuration

Set the **1-indexed** atom numbers (matching the order in your XYZ file) for:
- `ring_atoms`: the six ring atoms in connectivity order, e.g. `[O, C, C, C, C, C]`
- `forming_bond`: the two atoms whose distance should stay fixed during the search
  (usually the ring oxygen and the forming-bond carbon — same two atoms as the last
  entry in `ring_atoms`, i.e. `ring_atoms[0]` and `ring_atoms[-1]`)

Adjust these per structure — they do NOT have to be atoms 1-6; use whatever your
actual numbering is (e.g. 17-22 for the larger RR/RS scope structures).

In [ ]:
# ---- EDIT THESE FOR YOUR STRUCTURE ----
ring_atoms   = [17, 18, 19, 20, 21, 22]   # O-C-C-C-C-C in ring connectivity order
forming_bond = [17, 22]                   # atoms whose distance is held fixed
fc           = 1.0                        # constraint force constant (Hartree/Bohr^2)
method       = "gfn2"                     # "gfn2" or "gff" (GFN-FF, faster/rougher)
quick_mode   = True                       # True = faster/reduced sampling; False = thorough (slower)
threads      = 2
# -----------------------------------


## 5. Cremer–Pople ring-puckering tools

In [ ]:
import numpy as np

def read_xyz(path):
    with open(path) as f:
        lines = f.readlines()
    n = int(lines[0].strip())
    atoms = []
    for line in lines[2:2 + n]:
        parts = line.split()
        el = parts[0]
        x, y, z = map(float, parts[1:4])
        atoms.append((el, np.array([x, y, z])))
    return atoms

def cremer_pople_6(ring_coords):
    """Cremer, D.; Pople, J. A. JACS 1975, 97, 1354. Returns (Q, theta, phi) for a 6-ring."""
    R = np.asarray(ring_coords)
    N = 6
    Rm = R.mean(axis=0)
    Rp = R - Rm
    j = np.arange(N)
    Rprime  = (Rp * np.sin(2*np.pi*j/N)[:, None]).sum(axis=0)
    Rdouble = (Rp * np.cos(2*np.pi*j/N)[:, None]).sum(axis=0)
    n = np.cross(Rprime, Rdouble)
    n /= np.linalg.norm(n)
    z = Rp @ n
    c = np.sqrt(2/N) * np.sum(z * np.cos(2*2*np.pi*j/N))
    s = -np.sqrt(2/N) * np.sum(z * np.sin(2*2*np.pi*j/N))
    q2 = np.sqrt(c**2 + s**2)
    phi2 = np.degrees(np.arctan2(s, c)) % 360
    q3 = np.sqrt(1/N) * np.sum(z * ((-1)**j))
    Q = np.sqrt(q2**2 + q3**2)
    theta = np.degrees(np.arccos(np.clip(q3/Q, -1, 1))) if Q > 1e-9 else 0.0
    return Q, theta, phi2

def classify_6ring(theta, phi):
    if theta < 15 or theta > 165:
        return "Chair (C)"
    if 75 <= theta <= 105:
        m = phi % 60
        return "Boat (B)" if (m < 15 or m > 45) else "Twist-boat/Skew (S)"
    if 35 <= theta <= 65 or 115 <= theta <= 145:
        m = phi % 60
        return "Envelope (E)" if (m < 15 or m > 45) else "Half-chair (H)"
    return "Intermediate"

def dist(atoms, i, j):
    return np.linalg.norm(atoms[i-1][1] - atoms[j-1][1])

print("Cremer-Pople tools loaded.")


## 6. Run the constrained CREST conformer search (single structure)

In [ ]:
import subprocess, shutil

workdir = "/content/crest_run"
os.makedirs(workdir, exist_ok=True)
shutil.copy(xyz_path, os.path.join(workdir, "struc.xyz"))

toml = f"""input = 'struc.xyz'
runtype = 'imtd-gc'
threads = {threads}

[[calculation.level]]
method = "{method}"

[[calculation.constraint]]
type = 'bond'
atoms = [{forming_bond[0]}, {forming_bond[1]}]
fc = {fc}
"""
with open(os.path.join(workdir, "input.toml"), "w") as f:
    f.write(toml)

print(toml)

cmd = ["/content/crest/crest", "--input", "input.toml"]
if quick_mode:
    cmd.append("-mquick")

result = subprocess.run(cmd, cwd=workdir, capture_output=True, text=True, timeout=1800)
print(result.stdout[-4000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])


## 7. Classify every conformer CREST found, save to Drive

In [ ]:
import pandas as pd

conf_file = os.path.join(workdir, "crest_conformers.xyz")
lines = open(conf_file).readlines()
n_atoms = int(lines[0])
block = n_atoms + 2
n_structures = len(lines) // block

rows = []
for k in range(n_structures):
    chunk = lines[k*block:(k+1)*block]
    tmp_path = os.path.join(workdir, f"_tmp_conf_{k}.xyz")
    open(tmp_path, "w").writelines(chunk)
    atoms = read_xyz(tmp_path)

    energy = float(chunk[1].strip().split()[0])
    ring_coords = [atoms[i-1][1] for i in ring_atoms]
    Q, theta, phi = cremer_pople_6(ring_coords)
    cls = classify_6ring(theta, phi)
    d_form = dist(atoms, forming_bond[0], forming_bond[1])

    rows.append(dict(
        conformer=k+1, energy_Eh=energy,
        forming_bond_A=round(d_form, 3),
        Q=round(Q, 3), theta=round(theta, 1), phi=round(phi, 1),
        classification=cls,
    ))
    os.remove(tmp_path)

df = pd.DataFrame(rows)
df["rel_kcal"] = (df["energy_Eh"] - df["energy_Eh"].min()) * 627.5095
df = df.sort_values("rel_kcal").reset_index(drop=True)

out_csv = os.path.join(RESULTS_FOLDER, os.path.splitext(os.path.basename(xyz_path))[0] + "_pucker_results.csv")
df.to_csv(out_csv, index=False)
print("Saved to Drive:", out_csv)
df


## 8. Batch over all 34 structures in your Drive folder

Set per-file ring/forming-bond indices if they differ; otherwise this assumes every file
uses the same numbering set in Section 4. Results are written to Drive as you go, so if the
Colab runtime disconnects partway through a long batch you don't lose earlier results.

In [ ]:
def run_one(xyz_file, ring_atoms, forming_bond, fc=1.0, method="gfn2", quick_mode=True, threads=2, tag=None):
    tag = tag or os.path.splitext(os.path.basename(xyz_file))[0]
    wd = f"/content/crest_run_{tag}"
    os.makedirs(wd, exist_ok=True)
    shutil.copy(xyz_file, os.path.join(wd, "struc.xyz"))
    toml = f"""input = 'struc.xyz'
runtype = 'imtd-gc'
threads = {threads}

[[calculation.level]]
method = "{method}"

[[calculation.constraint]]
type = 'bond'
atoms = [{forming_bond[0]}, {forming_bond[1]}]
fc = {fc}
"""
    open(os.path.join(wd, "input.toml"), "w").write(toml)
    cmd = ["/content/crest/crest", "--input", "input.toml"] + (["-mquick"] if quick_mode else [])
    subprocess.run(cmd, cwd=wd, capture_output=True, text=True, timeout=1800)

    conf_file = os.path.join(wd, "crest_conformers.xyz")
    if not os.path.exists(conf_file):
        print("  WARNING: no conformers file for", tag)
        return pd.DataFrame()
    lines = open(conf_file).readlines()
    n_atoms = int(lines[0]); blk = n_atoms + 2
    out = []
    for k in range(len(lines)//blk):
        chunk = lines[k*blk:(k+1)*blk]
        tmp = os.path.join(wd, f"_tmp_{k}.xyz")
        open(tmp, "w").writelines(chunk)
        atoms = read_xyz(tmp)
        energy = float(chunk[1].strip().split()[0])
        Q, theta, phi = cremer_pople_6([atoms[i-1][1] for i in ring_atoms])
        out.append(dict(structure=tag, conformer=k+1, energy_Eh=energy,
                         forming_bond_A=round(dist(atoms, *forming_bond), 3),
                         Q=round(Q,3), theta=round(theta,1), phi=round(phi,1),
                         classification=classify_6ring(theta, phi)))
        os.remove(tmp)
    d = pd.DataFrame(out)
    if len(d):
        d["rel_kcal"] = (d["energy_Eh"] - d["energy_Eh"].min()) * 627.5095
    return d


# per-file overrides only if a file needs different indices than the Section 4 defaults; else leave empty
OVERRIDES = {
    # "RR_tBu.xyz": dict(ring_atoms=[17,18,19,20,21,22], forming_bond=[17,22]),
}

xyz_files = sorted(f for f in os.listdir(DRIVE_FOLDER) if f.endswith(".xyz"))
print(f"Found {len(xyz_files)} structures in {DRIVE_FOLDER}")

all_results = []
for fname in xyz_files:
    tag = os.path.splitext(fname)[0]
    cfg = OVERRIDES.get(fname, dict(ring_atoms=ring_atoms, forming_bond=forming_bond))
    print("Running:", tag)
    d = run_one(os.path.join(DRIVE_FOLDER, fname), fc=fc, method=method,
                quick_mode=quick_mode, threads=threads, tag=tag, **cfg)
    all_results.append(d)
    if len(d):
        d.to_csv(os.path.join(RESULTS_FOLDER, f"{tag}_pucker_results.csv"), index=False)

if all_results:
    batch_df = pd.concat(all_results, ignore_index=True)
    batch_df.to_csv(os.path.join(RESULTS_FOLDER, "ALL_34_pucker_results.csv"), index=False)
    print("Saved combined results to:", os.path.join(RESULTS_FOLDER, "ALL_34_pucker_results.csv"))
batch_df
